# 🕳️ AI-Based Pothole Detection — Complete Pipeline
**Model:** YOLOv8x + FPN neck | **Depth:** MiDaS | **Explainability:** GradCAM | **Mapping:** Folium/GPS

| Module | Method |
|---|---|
| Detection | YOLOv8x (largest YOLO8 variant, built-in FPN neck) |
| Size classification | Bounding-box area ratio |
| Severity grading | MiDaS inverse-depth score |
| Explainability | GradCAM heatmap on last C2f layer |
| GPS mapping | Folium + pixel-to-GPS projection |

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────
!pip install ultralytics kaggle opencv-python matplotlib scikit-learn folium grad-cam albumentations -q

In [ ]:
# ── CELL 2: Kaggle auth + dataset download ────────────────────
from google.colab import files
files.upload()   # upload kaggle.json

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d andrewmvd/pothole-detection
!unzip -q pothole-detection.zip
!ls

In [ ]:
# ── CELL 3: VOC XML → YOLO TXT labels ────────────────────────
import xml.etree.ElementTree as ET
import os

ANNOTATIONS_PATH = '/content/annotations'
IMAGES_PATH      = '/content/images'
OUTPUT_LABELS    = '/content/labels'
os.makedirs(OUTPUT_LABELS, exist_ok=True)

def convert(size, box):
    dw = 1. / size[0]; dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]; h = box[3] - box[2]
    return (x*dw, y*dh, w*dw, h*dh)

for xml_file in os.listdir(ANNOTATIONS_PATH):
    if not xml_file.endswith('.xml'): continue
    tree = ET.parse(os.path.join(ANNOTATIONS_PATH, xml_file))
    root = tree.getroot()
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)
    with open(os.path.join(OUTPUT_LABELS, xml_file.replace('.xml','.txt')), 'w') as out:
        for obj in root.iter('object'):
            xb = obj.find('bndbox')
            b  = (float(xb.find('xmin').text), float(xb.find('xmax').text),
                  float(xb.find('ymin').text), float(xb.find('ymax').text))
            bb = convert((w,h), b)
            out.write(f"0 {' '.join(map(str,bb))}\n")

print('✅ Labels converted:', len(os.listdir(OUTPUT_LABELS)), 'files')

In [ ]:
# ── CELL 4: Train / Val split ─────────────────────────────────
from sklearn.model_selection import train_test_split
import shutil, glob

images = glob.glob(IMAGES_PATH + '/*.png')
train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

def move(files, folder):
    os.makedirs(folder+'/images', exist_ok=True)
    os.makedirs(folder+'/labels', exist_ok=True)
    for p in files:
        name = os.path.basename(p)
        shutil.copy(p, folder+'/images/'+name)
        lbl = os.path.join(OUTPUT_LABELS, name.replace('.png','.txt'))
        if os.path.exists(lbl):
            shutil.copy(lbl, folder+'/labels/'+name.replace('.png','.txt'))

move(train_imgs, '/content/dataset/train')
move(val_imgs,   '/content/dataset/val')
print(f'✅ Train: {len(train_imgs)}  Val: {len(val_imgs)}')

In [ ]:
# ── CELL 5: YAML config ───────────────────────────────────────
with open('/content/pothole.yaml', 'w') as f:
    f.write("""
train: /content/dataset/train/images
val:   /content/dataset/val/images
nc: 1
names: ['pothole']
""")
print('✅ pothole.yaml ready')

In [ ]:
# ── CELL 6: Train YOLOv8x (FPN built-in, heavy augmentation) ──
# YOLOv8x uses a CSP backbone + PANet FPN neck — largest variant.
# Expected mAP@50 ≥ 0.92 after 150 epochs on this dataset.

from ultralytics import YOLO

model = YOLO('yolov8x.pt')

results = model.train(
    data         = '/content/pothole.yaml',
    epochs       = 150,
    imgsz        = 640,
    batch        = 16,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 5,
    # ── augmentation ──────────────────
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 10,
    translate    = 0.1,
    scale        = 0.5,
    shear        = 2.0,
    flipud       = 0.3,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.15,
    copy_paste   = 0.3,
    # ── checkpointing ─────────────────
    patience     = 30,
    save         = True,
    project      = '/content/runs/detect',
    name         = 'pothole_v2',
    exist_ok     = True,
    plots        = True,
    verbose      = True,
)
print('✅ Training complete')

In [ ]:
# ── CELL 7: Validation metrics ────────────────────────────────
from ultralytics import YOLO

BEST_PT = '/content/runs/detect/pothole_v2/weights/best.pt'
model   = YOLO(BEST_PT)
metrics = model.val(data='/content/pothole.yaml')

print('\n' + '='*45)
print('  VALIDATION RESULTS')
print('='*45)
print(f'  mAP@50      : {metrics.box.map50:.4f}')
print(f'  mAP@50-95   : {metrics.box.map:.4f}')
print(f'  Precision   : {metrics.box.mp:.4f}')
print(f'  Recall      : {metrics.box.mr:.4f}')
print('='*45)

In [ ]:
# ── CELL 8: GradCAM — visualise model attention ───────────────
import cv2, torch, numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

BEST_PT = '/content/runs/detect/pothole_v2/weights/best.pt'

class YOLOGradCAM:
    """GradCAM on the last C2f layer of YOLOv8 backbone."""
    def __init__(self, model_path):
        self.model = YOLO(model_path)
        self.gradients  = None
        self.activations = None
        self._hook()

    def _hook(self):
        target = None
        for layer in self.model.model.model:
            if hasattr(layer, 'cv2'):
                target = layer
        if target is None:
            target = list(self.model.model.model.children())[-2]
        target.register_forward_hook(
            lambda m,i,o: setattr(self, 'activations', o.detach()))
        target.register_full_backward_hook(
            lambda m,gi,go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, img_path):
        img_bgr = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        self.model.model.train()
        res = self.model(img_path)[0]
        if res.boxes is None or len(res.boxes) == 0:
            print('No detection — showing blank overlay.')
            self.model.model.eval()
            return img_rgb, img_rgb
        res.boxes.conf[res.boxes.conf.argmax()].backward(retain_graph=True)
        w   = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = (w * self.activations).sum(1).squeeze()
        cam = torch.relu(cam).cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam, (img_rgb.shape[1], img_rgb.shape[0]))
        hm  = cv2.applyColorMap((cam*255).astype(np.uint8), cv2.COLORMAP_JET)
        hm  = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(img_rgb, 0.55, hm, 0.45, 0)
        self.model.model.eval()
        return img_rgb, overlay

def show_gradcam(img_path):
    gcam = YOLOGradCAM(BEST_PT)
    orig, overlay = gcam.generate(img_path)
    fig, axes = plt.subplots(1,2,figsize=(14,5))
    axes[0].imshow(orig);    axes[0].set_title('Original');      axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title('GradCAM');       axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('/content/gradcam_output.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ GradCAM saved → /content/gradcam_output.png')

# Test on a val image (change path if needed)
import glob
sample = glob.glob('/content/dataset/val/images/*.png')[0]
show_gradcam(sample)

In [ ]:
# ── CELL 9: MiDaS Depth Estimation setup ─────────────────────
import torch

midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small')
midas.eval()
midas_transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
transform = midas_transforms.small_transform

def get_depth_map(img_rgb):
    inp = transform(img_rgb)
    with torch.no_grad():
        pred = midas(inp)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode='bicubic', align_corners=False
        ).squeeze()
    d = pred.cpu().numpy()
    return (d - d.min()) / (d.max() - d.min() + 1e-8) * 255

print('✅ MiDaS loaded')

In [ ]:
# ── CELL 10: Size + Severity classification helpers ───────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

BEST_PT = '/content/runs/detect/pothole_v2/weights/best.pt'
detector = YOLO(BEST_PT)

def classify_size(area, img_area):
    """Returns (label, BGR colour) by bounding-box area ratio."""
    r = area / img_area
    if r < 0.01:   return 'Small',  (0, 200,   0)   # green
    if r < 0.05:   return 'Medium', (0, 165, 255)   # orange
    return            'Large',  (0,   0, 255)   # red

def classify_severity(mean_depth):
    """MiDaS: lower value = deeper = more severe."""
    if mean_depth > 150:  return 'Low Severity'
    if mean_depth > 80:   return 'Medium Severity'
    return                       'High Severity'

def detect_and_annotate(img_path):
    img_pil = Image.open(img_path).convert('RGB')
    img_rgb = np.array(img_pil)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    depth   = get_depth_map(img_rgb)
    res     = detector(img_bgr)[0]
    H, W, _ = img_bgr.shape

    detections = []
    for box in res.boxes:
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        size_label, color = classify_size((x2-x1)*(y2-y1), H*W)
        sev = classify_severity(depth[y1:y2,x1:x2].mean())
        label = f'{size_label} | {sev} ({conf:.2f})'
        cv2.rectangle(img_bgr, (x1,y1),(x2,y2), color, 2)
        cv2.putText(img_bgr, label, (x1, max(y1-8,12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        detections.append({'size':size_label,'severity':sev,'conf':conf,
                            'cx_px':(x1+x2)/2,'cy_px':(y1+y2)/2,
                            'depth':float(depth[y1:y2,x1:x2].mean())})

    depth_vis = cv2.applyColorMap(depth.astype(np.uint8), cv2.COLORMAP_MAGMA)
    side = np.hstack([img_bgr, depth_vis])
    plt.figure(figsize=(16,6))
    plt.imshow(cv2.cvtColor(side, cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.title('Detection   |   Depth Map (MiDaS)')
    plt.tight_layout()
    plt.savefig('/content/detection_depth.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ {len(detections)} pothole(s) detected')
    return detections, (H, W)

In [ ]:
# ── CELL 11: GPS Mapping with Folium ──────────────────────────
import folium, math

def pixel_to_gps(cx_px, cy_px, H, W,
                 origin_lat, origin_lon,
                 fov_deg=60, alt_m=2.0):
    """Converts bounding-box centre pixel to approximate GPS coords."""
    half   = math.radians(fov_deg/2)
    gw     = alt_m * math.tan(half)          # ground half-width (m)
    gh     = gw * (H/W)                      # ground half-height (m)
    dx_m   = ((cx_px/W) - 0.5)*2*gw
    dy_m   = (0.5 - (cy_px/H))*2*gh
    dlat   = dy_m / 111_320
    dlon   = dx_m / (111_320 * math.cos(math.radians(origin_lat)))
    return origin_lat+dlat, origin_lon+dlon

def build_map(detections, H, W,
              origin_lat=12.9716, origin_lon=77.5946,
              output='/content/pothole_map.html'):
    sev_color = {'Low Severity':'green','Medium Severity':'orange','High Severity':'red'}
    m = folium.Map(location=[origin_lat, origin_lon], zoom_start=18)
    for i, d in enumerate(detections):
        lat, lon = pixel_to_gps(d['cx_px'],d['cy_px'],H,W,origin_lat,origin_lon)
        c = sev_color.get(d['severity'],'blue')
        folium.CircleMarker(
            location=[lat,lon], radius=10,
            color=c, fill=True, fill_color=c, fill_opacity=0.85,
            popup=folium.Popup(
                f"<b>Pothole #{i+1}</b><br>Size: {d['size']}<br>"
                f"Severity: {d['severity']}<br>Conf: {d['conf']:.2f}<br>"
                f"GPS: {lat:.6f}, {lon:.6f}", max_width=250),
            tooltip=f"#{i+1} {d['size']} | {d['severity']}"
        ).add_to(m)
    legend = """
    <div style="position:fixed;bottom:30px;left:30px;z-index:1000;
                background:white;padding:10px;border-radius:8px;
                border:2px solid grey;font-size:13px;">
      <b>Pothole Severity</b><br>
      <span style='color:green;'>&#9679;</span> Low<br>
      <span style='color:orange;'>&#9679;</span> Medium<br>
      <span style='color:red;'>&#9679;</span> High
    </div>"""
    m.get_root().html.add_child(folium.Element(legend))
    m.save(output)
    print(f'✅ Map saved → {output}')
    return m

In [ ]:
# ── CELL 12: Full Pipeline — upload your own image ────────────
from google.colab import files
from IPython.display import display

uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# ✏️  Set GPS coordinates where the photo was taken
CAPTURE_LAT = 12.9716   # e.g., Bengaluru
CAPTURE_LON = 77.5946

# 1. Detect + depth
detections, (H, W) = detect_and_annotate(img_path)

# 2. GradCAM
show_gradcam(img_path)

# 3. GPS map
m = build_map(detections, H, W, CAPTURE_LAT, CAPTURE_LON)
display(m)

# 4. Summary table
print('\n' + '='*55)
print(f'  {"#":<4} {"Size":<8} {"Severity":<18} {"Conf":<6} {"Depth"}')
print('='*55)
for i,d in enumerate(detections):
    print(f'  {i+1:<4} {d["size"]:<8} {d["severity"]:<18} {d["conf"]:.2f}   {d["depth"]:.1f}')
print('='*55)

In [ ]:
# ── CELL 13 (optional): Export model to ONNX ─────────────────
from ultralytics import YOLO
model = YOLO('/content/runs/detect/pothole_v2/weights/best.pt')
model.export(format='onnx', imgsz=640)
print('✅ ONNX model exported')